<a href="https://colab.research.google.com/github/RealHotnCold/MLA0204-192511036/blob/main/3_ID3_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ID3 Decision Tree Demonstration
This notebook demonstrates the working of the **ID3 (Iterative Dichotomiser 3)** decision tree algorithm using the `play_tennis.csv` dataset.

### ID3 Key Concepts:
1. **Entropy**: Measures the impurity or randomness in the dataset.
   $$H(S) = -\sum p_i \log_2 p_i$$
2. **Information Gain**: Measures the reduction in entropy after splitting the dataset on an attribute $A$.
   $$Gain(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

In [1]:
import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_csv('/content/play_tennis.csv')
display(df)

,Outlook,Temperature,Humidity,Wind,Target
0,sunny,hot,high,weak,no
1,sunny,hot,high,strong,no
2,overcast,hot,high,weak,yes
3,rain,mild,high,weak,yes
4,rain,cool,normal,weak,yes
5,rain,cool,normal,strong,no
6,overcast,cool,normal,strong,yes
7,sunny,mild,high,weak,no
8,sunny,cool,normal,weak,yes
9,rain,mild,normal,weak,yes


### ID3 Algorithm Implementation
Let's implement the core helper functions to calculate **Entropy**, **Information Gain**, and recursively build the decision tree structure.

In [2]:
def calculate_entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy = 0
    for i in range(len(elements)):
        p = counts[i] / np.sum(counts)
        entropy -= p * np.log2(p)
    return entropy

def calculate_info_gain(data, split_attribute_name, target_name):
    total_entropy = calculate_entropy(data[target_name])

    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = 0
    for i in range(len(vals)):
        subset = data[data[split_attribute_name] == vals[i]]
        prob = counts[i] / np.sum(counts)
        weighted_entropy += prob * calculate_entropy(subset[target_name])

    information_gain = total_entropy - weighted_entropy
    return information_gain

def id3(data, original_data, features, target_attribute_name="play", parent_node_class=None):
    # 1. If all target values are the same, return this value
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # 2. If the dataset is empty, return the mode target value in the original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])]

    # 3. If feature space is empty, return the parent node class
    elif len(features) == 0:
        return parent_node_class

    else:
        # Set default parent node class
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])]

        # Select the feature that best splits the data
        item_values = [calculate_info_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree structure
        tree = {best_feature: {}}

        # Remove the best feature from the feature space
        remaining_features = [i for i in features if i != best_feature]

        # Grow a branch for each unique value of the best feature
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, data, remaining_features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

### Build and Train the Decision Tree
Now we will prepare the dataset features and execute the ID3 algorithm.

In [3]:
# Identify target and feature columns (excluding identifiers like 'day' if present)
target_col = 'play'
if 'play' not in df.columns:
    # Fallback to last column if 'play' is named differently (e.g., 'PlayTennis' or 'Play')
    target_col = df.columns[-1]

feature_cols = [col for col in df.columns if col not in [target_col, 'day', 'Day']]

# Build the tree
tree = id3(df, df, feature_cols, target_attribute_name=target_col)

# Display the tree representation
import pprint
pprint.pprint(tree)

{'Outlook': {'overcast': 'yes',
             'rain': {'Wind': {'strong': 'no', 'weak': 'yes'}},
             'sunny': {'Humidity': {'high': 'no', 'normal': 'yes'}}}}
